In [10]:
import asyncio
from fastmcp import Client
import json
from openai import OpenAI

In [54]:
# HTTP server
client = Client("http://localhost:8000/mcp")
client_openai=OpenAI()

In [171]:
def print_completion_info(response):
    print(f"Why it stopped: {response.choices[0].finish_reason}")
    print(f"Message: {response.choices[0].message.content}")
    
    tool_calls = response.choices[0].message.tool_calls
    if tool_calls:
        for tool_call in tool_calls:
            print(f"Function call: {tool_call.function.name}")
            print(f"Inputs: {tool_call.function.arguments}")
    else:
        print("Function call: None")

# Usage:
# print_completion_info(response)

---------------------

### Trying tool

In [58]:
async def call_tool(tool, payload):
    
    async with client:
        result = await client.call_tool(tool,payload)
    return result

In [146]:
# Then call it with await:
result = await call_tool("greet", {"name": "Ford"})

In [60]:
print(f"\n Output: {result.structured_content}")


 Output: {'result': 'Hello, Ford!'}


-------------------



### Calling ChatGPT

In [61]:
response = client_openai.responses.create(model="gpt-4o", 
                                   input="Hi are you there?")
# response

In [62]:
response.output[0].content[0].text

"Hello! Yes, I'm here. How can I assist you today?"

--------

### Listint tools

In [112]:
async with client:
    tools = await client.list_tools()
# print(tools)

In [162]:
async def mcp_to_openai_tools():
    async with client:
        mcp_tools = await client.list_tools()
        tools = [{
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description,
                "parameters": {
                    "type": "object",
                    "properties": tool.inputSchema.get("properties", {}),
                    "required": tool.inputSchema.get("required", []),
                    "additionalProperties": False
                },
                "strict": True
            }
        }
        for tool in mcp_tools]


    print("Converted tools:", tools)
    return tools

In [164]:
# Then call it with await:
mcp_tools = await mcp_to_openai_tools()

Converted tools: [{'type': 'function', 'function': {'name': 'greet', 'description': 'Greet a person by name with a friendly hello message.\n\nThis tool generates a personalized greeting for the given name.', 'parameters': {'type': 'object', 'properties': {'name': {'description': 'The name of the person to greet', 'type': 'string'}}, 'required': ['name'], 'additionalProperties': False}, 'strict': True}}, {'type': 'function', 'function': {'name': 'add_numbers', 'description': 'Add two numbers together and return the sum.\n\nThis tool performs basic addition of two integer values.', 'parameters': {'type': 'object', 'properties': {'a': {'description': 'The first number to add', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'type': 'integer'}}, 'required': ['a', 'b'], 'additionalProperties': False}, 'strict': True}}]


-----------------------------------------

### Agent

In [115]:
system_prompt= "Your are an assisstant that must try to answer the user question using one of the tools. Tell me what tool can be useful, or say that any tool is useful for the task."

In [116]:
prompt="I need to add to a ticket a cost of a item. Ticket is 40, and the item to add cost 2"

In [117]:
input_messages = [{"role": "system", "content":system_prompt },
                  {"role": "user", "content": prompt}]


In [166]:
response = client_openai.chat.completions.create(
    model="gpt-4o",
    messages=input_messages,  # Note: 'messages' not 'input'
    tools=mcp_tools
)

In [167]:
response

ChatCompletion(id='chatcmpl-D24gxHKOyLH5IKvMBOYA1UqyIk1Ua', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Km4anqqI4A6U8iiJA0ZlaQP6', function=Function(arguments='{"a":40,"b":2}', name='add_numbers'), type='function')]))], created=1769387063, model='gpt-4o-2024-08-06', object='chat.completion', service_tier='default', system_fingerprint='fp_deacdd5f6f', usage=CompletionUsage(completion_tokens=18, prompt_tokens=183, total_tokens=201, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [172]:
print_completion_info(response)

Why it stopped: tool_calls
Message: None
Function call: add_numbers
Inputs: {"a":40,"b":2}


In [199]:
response.choices[0].message.tool_calls[0]

ChatCompletionMessageFunctionToolCall(id='call_Km4anqqI4A6U8iiJA0ZlaQP6', function=Function(arguments='{"a":40,"b":2}', name='add_numbers'), type='function')

In [198]:
tool_reponse = response.choices[0].message.tool_calls[0]

In [197]:
tool_reponse.type

'function'

In [206]:
response.choices[0].message.tool_calls[0]

ChatCompletionMessageFunctionToolCall(id='call_Km4anqqI4A6U8iiJA0ZlaQP6', function=Function(arguments='{"a":40,"b":2}', name='add_numbers'), type='function')

In [213]:
response.choices[0].message.tool_calls[0].function

Function(arguments='{"a":40,"b":2}', name='add_numbers')

In [217]:
tool_call = response.choices[0].message.tool_calls[0].function
tool_call

Function(arguments='{"a":40,"b":2}', name='add_numbers')

In [222]:
# Step 2: Handle function calls
if response.choices[0].finish_reason == "tool_calls":
    for args, name in response.choices[0].message.tool_calls[0].function:
        # Step 3: Execute function
        # args = json.loads(tool_call.arguments)
        if name == "get_weather":
            result = get_weather(**args)
        
        # Step 4: Append function call and result to messages
        input_messages.append(tool_call)
        input_messages.append(
            {
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": str(result),
            }
        )

In [219]:


        # Step 4: Append function call and result to messages
        input_messages.append(tool_call)
        input_messages.append(
            {
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": str(result),
            }
        )

('arguments', '{"a":40,"b":2}')


NameError: name 'call_function' is not defined